# Preprocessing in scikit-learn: Complete Guide
Run cells top to bottom (Runtime > Run all). Colab already has everything installed.

**Standard order:** Split > Impute > Encode > Scale/Transform > Model
**Golden rule:** `fit` on train only, `transform` on train and test.

In [1]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.impute import SimpleImputer, KNNImputer, MissingIndicator
from sklearn.experimental import enable_iterative_imputer   # required (still experimental)
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler, MaxAbsScaler, Normalizer,
    OneHotEncoder, OrdinalEncoder, LabelEncoder, TargetEncoder,
    PowerTransformer, QuantileTransformer, FunctionTransformer,
    Binarizer, KBinsDiscretizer, PolynomialFeatures,
)
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

print("scikit-learn version:", sklearn.__version__)   # TargetEncoder needs 1.3+, sparse_output needs 1.2+

scikit-learn version: 1.6.1


## 0. Sample dataset (with NaN, outliers, skew, categories)

In [2]:
rng = np.random.default_rng(42)
n = 300
df = pd.DataFrame({
    "age":    rng.integers(18, 65, n).astype(float),
    "salary": rng.lognormal(10.5, 0.5, n),                 # right-skewed
    "city":   rng.choice(["Karachi", "Lahore", "Islamabad", "Quetta", "Multan"], n, p=[.4, .3, .15, .1, .05]),
    "edu":    rng.choice(["BS", "MS", "PhD"], n, p=[.5, .35, .15]),
})
df.loc[0, "salary"] = 5_000_000                            # outlier
df["bought"] = ((df["age"] > 35) ^ (df["edu"] == "PhD") | (df["salary"] > 45000)).astype(int)

# inject missing values
for col, frac in [("age", .08), ("salary", .08), ("city", .06)]:
    df.loc[rng.choice(n, int(n * frac), replace=False), col] = np.nan

X = df.drop(columns="bought")
y = df["bought"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(X_train.shape, X_test.shape)
print(X_train.isna().sum())
display(X_train.head())

(240, 4) (60, 4)
age       20
salary    17
city      15
edu        0
dtype: int64


,age,salary,city,edu
46,38.0,20878.540894,Karachi,MS
67,35.0,59956.759875,Karachi,MS
195,22.0,44664.985106,Karachi,BS
216,21.0,34115.103559,Lahore,BS
5,58.0,28476.829781,Karachi,MS


## 1. Missing values (imputation)
| Tool | How it fills |
|---|---|
| `SimpleImputer` | One statistic per column |
| `KNNImputer` | Average of k most similar rows |
| `IterativeImputer` | Predicts each column from the others |
| `MissingIndicator` | Adds 0/1 "was missing" columns |

In [3]:
num_cols = ["age", "salary"]

# SimpleImputer: median is safe with outliers
si = SimpleImputer(strategy="median")
si.fit(X_train[num_cols])
print("learned medians:", si.statistics_)

# categorical: most_frequent or constant
print("mode            :", SimpleImputer(strategy="most_frequent").fit(X_train[["city"]]).statistics_)
print("constant        :", SimpleImputer(strategy="constant", fill_value="Unknown")
      .fit_transform(X_train[["city"]])[:5].ravel())

# fill + indicator columns
ind = SimpleImputer(strategy="median", add_indicator=True).fit_transform(X_train[num_cols])
print("with indicator shape:", ind.shape, "(2 numeric + 2 indicator columns)")

# MissingIndicator alone
mi = MissingIndicator().fit(X_train[num_cols])
print("indicator features:", mi.features_)

learned medians: [   40.         37293.62478714]
mode            : ['Karachi']
constant        : ['Karachi' 'Karachi' 'Karachi' 'Lahore' 'Karachi']
with indicator shape: (240, 4) (2 numeric + 2 indicator columns)
indicator features: [0 1]


In [4]:
# KNNImputer (scale first: it is distance-based)
sc = StandardScaler().fit(X_train[num_cols])
scaled = sc.transform(X_train[num_cols])
knn = KNNImputer(n_neighbors=5).fit(scaled)
filled = sc.inverse_transform(knn.transform(scaled))
print("KNN has NaN left:", np.isnan(filled).any())

# IterativeImputer
it = IterativeImputer(random_state=0).fit(X_train[num_cols])
print("Iterative has NaN left:", np.isnan(it.transform(X_train[num_cols])).any())

KNN has NaN left: False
Iterative has NaN left: False


## 2. Scaling
| Tool | Formula | Best for |
|---|---|---|
| `StandardScaler` | (x - mean) / std | Default |
| `MinMaxScaler` | (x - min) / (max - min) | Neural nets, bounded range |
| `RobustScaler` | (x - median) / IQR | Outliers |
| `MaxAbsScaler` | x / max(abs(x)) | Sparse data |
| `Normalizer` | scales each **row** to length 1 | Text, cosine similarity |

In [5]:
clean = SimpleImputer(strategy="median").fit_transform(X_train[num_cols])
clean_df = pd.DataFrame(clean, columns=num_cols)

summary = {}
for name, scaler in [("Standard", StandardScaler()), ("MinMax", MinMaxScaler()),
                     ("Robust", RobustScaler()), ("MaxAbs", MaxAbsScaler())]:
    out = scaler.fit_transform(clean_df)
    summary[name] = {"min": out.min(), "max": out.max(), "mean": out.mean().round(3), "std": out.std().round(3)}
display(pd.DataFrame(summary).T)

print("Normalizer row norms:", np.linalg.norm(Normalizer().fit_transform(clean_df)[:3], axis=1))

,min,max,mean,std
Standard,-1.864590,4.246131,-0.000,1.000
MinMax,0.000000,1.000000,0.377,0.250
Robust,-1.350347,4.491494,0.108,0.850
MaxAbs,0.078704,1.000000,0.477,0.235


Normalizer row norms: [1. 1. 1.]


In [6]:
# StandardScaler by hand
a = np.array([[20], [30], [40], [50]])
print("by hand:", ((a - a.mean()) / a.std()).ravel())
print("sklearn:", StandardScaler().fit_transform(a).ravel())

# Why Robust beats Standard with an outlier: look at the typical (non-outlier) values
std_out = StandardScaler().fit_transform(clean_df[["salary"]]).ravel()
rob_out = RobustScaler().fit_transform(clean_df[["salary"]]).ravel()
print("salary spread of typical rows -> Standard:", np.percentile(std_out, [25, 75]).round(2),
      "| Robust:", np.percentile(rob_out, [25, 75]).round(2))

by hand: [-1.34164079 -0.4472136   0.4472136   1.34164079]
sklearn: [-1.34164079 -0.4472136   0.4472136   1.34164079]
salary spread of typical rows -> Standard: [-0.67  0.32] | Robust: [-0.49  0.51]


## 3. Encoding categorical features
| Tool | Use for |
|---|---|
| `OneHotEncoder` | Unordered categories |
| `OrdinalEncoder` | Ordered categories |
| `LabelEncoder` | Target `y` only |
| `TargetEncoder` | High-cardinality categories |

In [7]:
city_filled = SimpleImputer(strategy="most_frequent").fit_transform(X_train[["city"]])
city_test = SimpleImputer(strategy="most_frequent").fit(X_train[["city"]]).transform(X_test[["city"]])

ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
out = ohe.fit_transform(city_filled)
print("OneHot features:", ohe.get_feature_names_out())

# unseen category at prediction time does not crash
print("unseen city ->", ohe.transform(pd.DataFrame({"x": ["Peshawar"]}).values))

print("drop='first'    :", OneHotEncoder(drop="first", sparse_output=False).fit(city_filled).get_feature_names_out())
print("min_frequency   :", OneHotEncoder(min_frequency=0.1, sparse_output=False).fit(city_filled).get_feature_names_out())

OneHot features: ['x0_Islamabad' 'x0_Karachi' 'x0_Lahore' 'x0_Multan' 'x0_Quetta']
unseen city -> [[0. 0. 0. 0. 0.]]
drop='first'    : ['x0_Karachi' 'x0_Lahore' 'x0_Multan' 'x0_Quetta']
min_frequency   : ['x0_Islamabad' 'x0_Karachi' 'x0_Lahore' 'x0_infrequent_sklearn']


In [8]:
oe = OrdinalEncoder(categories=[["BS", "MS", "PhD"]])      # you define the order
print("Ordinal:", oe.fit_transform(X_train[["edu"]]).ravel()[:8])

oe_safe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1).fit(X_train[["edu"]])
print("unseen edu ->", oe_safe.transform(pd.DataFrame({"edu": ["Diploma"]})))

le = LabelEncoder()                                         # target only, never X
print("LabelEncoder:", le.fit_transform(["cat", "dog", "cat"]), le.classes_)

# TargetEncoder (sklearn 1.3+)
te = TargetEncoder(random_state=0)
te_out = te.fit_transform(city_filled, y_train)
print("TargetEncoder (cross-fitted) sample:", te_out[:5].ravel().round(3))

Ordinal: [1. 1. 0. 0. 1. 0. 0. 1.]
unseen edu -> [[-1.]]
LabelEncoder: [0 1 0] ['cat' 'dog']
TargetEncoder (cross-fitted) sample: [0.718 0.69  0.713 0.825 0.718]


## 4. Changing distributions and creating features

In [9]:
sal = clean_df[["salary"]]
print("skew before:", sal["salary"].skew().round(2))
print("PowerTransformer :", pd.Series(PowerTransformer().fit_transform(sal).ravel()).skew().round(2))
print("QuantileTransformer:", pd.Series(QuantileTransformer(output_distribution="normal", n_quantiles=100)
      .fit_transform(sal).ravel()).skew().round(2))
print("log1p            :", pd.Series(np.asarray(FunctionTransformer(np.log1p).fit_transform(sal)).ravel()).skew().round(2))

print("Binarizer        :", Binarizer(threshold=40000).fit_transform(sal)[:8].ravel())
print("KBinsDiscretizer :", KBinsDiscretizer(n_bins=4, encode="ordinal", strategy="quantile")
      .fit_transform(sal)[:8].ravel())
print("PolynomialFeatures:\n", PolynomialFeatures(degree=2, include_bias=False)
      .fit_transform(clean_df[["age"]].head(3)))

skew before: 1.36
PowerTransformer : 0.0
QuantileTransformer: 0.01
log1p            : -0.07
Binarizer        : [0. 1. 1. 0. 0. 0. 0. 1.]
KBinsDiscretizer : [0. 3. 2. 1. 1. 1. 0. 3.]
PolynomialFeatures:
 [[  38. 1444.]
 [  35. 1225.]
 [  22.  484.]]


## 5. ColumnTransformer + Pipeline (the professional way)
`Pipeline` chains steps. `ColumnTransformer` applies different steps to different columns.

In [10]:
num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler()),
])
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),   # dense so pandas output works
])
ord_pipe = Pipeline([
    ("ord", OrdinalEncoder(categories=[["BS", "MS", "PhD"]])),
])

prep = ColumnTransformer([
    ("num", num_pipe, ["age", "salary"]),
    ("cat", cat_pipe, ["city"]),
    ("ord", ord_pipe, ["edu"]),
])

model = Pipeline([("prep", prep), ("clf", LogisticRegression(max_iter=500))])
model.fit(X_train, y_train)          # preprocessing is fit on train automatically
print("test accuracy:", round(model.score(X_test, y_test), 3))
print("feature names:", prep.get_feature_names_out())

test accuracy: 0.75
feature names: ['num__age' 'num__salary' 'cat__city_Islamabad' 'cat__city_Karachi'
 'cat__city_Lahore' 'cat__city_Multan' 'cat__city_Quetta' 'ord__edu']


In [11]:
# Auto-select columns by dtype instead of typing names
prep_auto = ColumnTransformer([
    ("num", num_pipe, make_column_selector(dtype_include=np.number)),
    ("cat", cat_pipe, make_column_selector(dtype_exclude=np.number)),   # city + edu as one-hot
], remainder="drop")                 # "passthrough" would keep untouched columns

auto_model = Pipeline([("prep", prep_auto), ("clf", LogisticRegression(max_iter=500))])
auto_model.fit(X_train, y_train)
print("auto-select accuracy:", round(auto_model.score(X_test, y_test), 3))

# keep DataFrame output
prep.set_output(transform="pandas")
display(prep.fit_transform(X_train).head())
prep.set_output(transform="default")

auto-select accuracy: 0.767


,num__age,num__salary,cat__city_Islamabad,cat__city_Karachi,cat__city_Lahore,cat__city_Multan,cat__city_Quetta,ord__edu
46,-0.105263,-0.814027,0.0,1.0,0.0,0.0,0.0,1.0
67,-0.263158,1.123869,0.0,1.0,0.0,0.0,0.0,1.0
195,-0.947368,0.365547,0.0,1.0,0.0,0.0,0.0,0.0
216,-1.000000,-0.157623,0.0,0.0,1.0,0.0,0.0,0.0
5,0.947368,-0.437227,0.0,1.0,0.0,0.0,0.0,1.0


ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', RobustScaler())]),
                                 ['age', 'salary']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('ohe',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['city']),
                                ('ord',
                                 Pipeline(steps=[('ord',
                                                  OrdinalEncoder(categories=[['BS',
                                                                              'MS',
                                                                              'PhD']]))]),
                                 ['edu'])])

## 6. Cross-validation and tuning preprocessing together with the model

In [12]:
model = Pipeline([("prep", prep), ("clf", LogisticRegression(max_iter=500))])

# Cross-validation: preprocessing is re-fit inside each fold (no leakage)
scores = cross_val_score(model, X_train, y_train, cv=5)
print("CV accuracy:", scores.round(3), "| mean:", scores.mean().round(3))

# Tune preprocessing + model together: step__substep__param
grid = GridSearchCV(model, {
    "prep__num__imputer__strategy": ["mean", "median"],
    "clf__C": [0.1, 1, 10],
}, cv=5)
grid.fit(X_train, y_train)
print("best params:", grid.best_params_)
print("best CV score:", round(grid.best_score_, 3))
print("test accuracy:", round(grid.score(X_test, y_test), 3))

CV accuracy: [0.771 0.854 0.75  0.771 0.771] | mean: 0.783
best params: {'clf__C': 1, 'prep__num__imputer__strategy': 'mean'}
best CV score: 0.792
test accuracy: 0.75


## 7. Outliers

In [13]:
# Option 1: keep, use robust tools (RobustScaler, median imputation)
# Option 2: reduce skew (PowerTransformer, log1p)
# Option 3: cap values (winsorize) inside a pipeline
def cap_outliers(x):
    lo, hi = np.nanpercentile(x, [1, 99], axis=0)
    return np.clip(x, lo, hi)

capper = FunctionTransformer(cap_outliers)
raw = X_train[["salary"]].to_numpy(dtype=float)
print("max before cap:", np.nanmax(raw).round(0), "| after cap:", np.nanmax(capper.fit_transform(raw)).round(0))

max before cap: 127866.0 | after cap: 105297.0


## 8. Common mistake demo: data leakage

In [14]:
vals = X_train[["salary"]].fillna(X_train["salary"].median())
test_vals = X_test[["salary"]].fillna(X_train["salary"].median())

good = StandardScaler().fit(vals)          # correct: learn from TRAIN
bad = StandardScaler().fit(test_vals)      # WRONG: learn from TEST
print("correct :", good.transform(test_vals).ravel()[:4].round(3))
print("WRONG   :", bad.transform(test_vals).ravel()[:4].round(3))

# Another classic mistake: 1D Series into a transformer
try:
    SimpleImputer().fit_transform(X_train["age"])
except ValueError as e:
    print("\n1D input error ->", str(e)[:70], "...")

correct : [-0.831 -0.855 -1.119 -0.089]
WRONG   : [-0.154 -0.155 -0.163 -0.13 ]

1D input error -> Expected a 2-dimensional container but got <class 'pandas.core.series. ...


## 9. Practice: Titanic
1. Load: `import seaborn as sns; t = sns.load_dataset("titanic")`
2. Target: `survived`. Features: `age, fare, sex, embarked, class`
3. Impute `age` and `embarked`, one-hot `sex` and `embarked`, ordinal `class` (Third < Second < First), scale `age` and `fare`
4. Build `ColumnTransformer` + `Pipeline` + `LogisticRegression`
5. Report test accuracy and 5-fold CV score